In [1]:
import json
import pandas as pd
import re
from IPython.display import display
from tqdm import tqdm
tqdm.pandas()
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

[nltk_data] Downloading package punkt to /Users/justdann/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/justdann/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/justdann/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("../Data/Raw/Case 2 Dataset.csv")
display(df.head())
df.info()
print(df.isnull().sum())

,date,lang,source,user_id,hashtags,is_quote,mentions,tweet_id,username,verified,...,quoted_tweet_id,quoted_username,user_created_at,user_description,in_reply_to_user_id,user_statuses_count,user_followers_count,user_following_count,in_reply_to_status_id,in_reply_to_screen_name
0,"45662,58063",in,dlvr.it,"1,6663E+18",[],False,[],"1,8759E+18",arinadotid,False,...,NaN,NaN,Wed Jun 07 04:40:18 +0000 2023,Media khas keislaman yang berjejaring dengan m...,NaN,5911.0,110.0,20.0,NaN,NaN
1,"45752,56706",in,Twitter Web App,"1,35691E+18","[""PenuhiGiziIndonesia"",""Indonesiaemas"",""AstaCi...",False,[],"1,90851E+18",ArifaNidya,False,...,NaN,NaN,Wed Feb 03 10:18:32 +0000 2021,make me special,NaN,5069.0,15.0,25.0,NaN,NaN
2,"45671,93193",in,Twitter for Android,553451866,[],False,"[{""name"":""Lambe Waras"",""user_id"":""2905814730"",...","1,87929E+18",ZainAris,False,...,NaN,NaN,Sat Apr 14 11:15:29 +0000 2012,Pemerhati sospol hukum dan ekonomi,2905814730,19762.0,701.0,1103.0,"1,87929E+18",abu_waras
3,"45715,64344",in,Twitter for Android,"1,30075E+18",[],False,"[{""name"":""Tanyarl 💚"",""user_id"":""13316505595189...","1,89513E+18",hereseans,False,...,NaN,NaN,Tue Sep 01 10:52:47 +0000 2020,INTJ,"1,75087E+18",20947.0,1977.0,2052.0,"1,89513E+18",HukumItb
4,"45946,96626",in,Twitter for Android,"1,91004E+18",[],False,[],"1,97896E+18",revirevii1,False,...,NaN,NaN,Wed Apr 09 18:43:39 +0000 2025,NaN,NaN,1198.0,1002.0,1116.0,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     15000 non-null  str    
 1   lang                     15000 non-null  str    
 2   source                   15000 non-null  str    
 3   user_id                  15000 non-null  str    
 4   hashtags                 15000 non-null  str    
 5   is_quote                 15000 non-null  bool   
 6   mentions                 15000 non-null  str    
 7   tweet_id                 15000 non-null  str    
 8   username                 15000 non-null  str    
 9   verified                 15000 non-null  bool   
 10  full_text                15000 non-null  str    
 11  created_at               15000 non-null  str    
 12  quoted_url               1991 non-null   str    
 13  view_count               15000 non-null  int64  
 14  quote_count              15000 no

In [3]:
from Src.stopwords_id import get_stopwords
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()
indonesian_stopwords = get_stopwords()

In [4]:
slang_url = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
slang_df = pd.read_csv(slang_url)
print(slang_df.head())
print(slang_df.columns.tolist())

     slang    formal  In-dictionary  \
0     woww       wow              1   
1    aminn      amin              1   
2      met   selamat              1   
3   netaas   menetas              1   
4  keberpa  keberapa              0   

                                             context  category1 category2  \
0                                                wow   elongasi         0   
1  Selamat ulang tahun kakak tulus semoga panjang...   elongasi         0   
2  Met hari netaas kak!? Wish you all the best @t...  abreviasi         0   
3  Met hari netaas kak!? Wish you all the best @t...   afiksasi  elongasi   
4                           Birthday yg keberpa kak?  abreviasi         0   

  category3  
0         0  
1         0  
2         0  
3         0  
4         0  
['slang', 'formal', 'In-dictionary', 'context', 'category1', 'category2', 'category3']


In [5]:
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

# Cek
print(slang_dict.get('anjir'))

None


In [6]:
custom_slang = {
    # Kata Ganti & Sapaan
    "gw": "saya", "gweh": "saya", "gue": "saya",
    "lu": "kamu", "loe": "kamu", "elu": "kamu",
    "ortu": "orang tua", "kepsek": "kepala sekolah",
    "bumil": "ibu hamil", "busui": "ibu menyusui",

    # Negasi & Konjungsi
    "ga": "tidak", "gak": "tidak", "gk": "tidak", "ngga": "tidak", "nggak": "tidak", "kaga": "tidak",
    "yg": "yang", "pd": "pada", "dgn": "dengan", "utk": "untuk",
    "dr": "dari", "drpd": "daripada", "krn": "karena", "karna": "karena",
    "tp": "tapi", "tpi": "tapi", "sm": "sama", "sma": "sama", "dan": "dan",
    "klo": "kalau", "klu": "kalau", "klw": "kalau",

    # Keterangan Waktu & Sifat
    "sdh": "sudah", "udh": "sudah", "udah": "sudah",
    "blm": "belum", "blom": "belum",
    "skrg": "sekarang", "td": "tadi", "bsk": "besok", "kmrn": "kemarin",
    "dl": "dulu", "dlu": "dulu",
    "trs": "terus", "trus": "terus",
    "bgt": "banget", "bngt": "banget", "bgttttt": "banget",
    "aja": "saja", "aje": "saja", "ae": "saja",

    # Kata Kerja & Kata Benda
    "bikin": "membuat", "pake": "pakai", "dpt": "dapat",
    "bnyk": "banyak", "bbrp": "beberapa", "cmn": "cuma",
    "duit": "uang", "dwit": "uang", "sklh": "sekolah",
    "tau": "tahu", "tw": "tahu", "bener": "benar", "bner": "benar",
    "jd": "jadi", "jdi": "jadi", "bkn": "bukan",
    "sampe": "sampai", "smpe": "sampai",
    "pdhl": "padahal", "jg": "juga", "jga": "juga",

    # Kata Tanya & Penunjuk
    "gmn": "bagaimana", "gimana": "bagaimana", "gmna": "bagaimana", "bgmn": "bagaimana",
    "knp": "kenapa", "knpa": "kenapa",
    "gitu": "begitu", "bgtu": "begitu",
    "gini": "begini", "kek": "seperti", "kyk": "seperti",

    # Umpatan / Ekspresi Ekstrem (Bisa disesuaikan atau dibuang)
    "anjir": "anjing", "ajg": "anjing", "bjir": "anjing", "njir": "anjing", "jir": "anjing",
    "gblk": "goblok", "bego": "bodoh",

    #bahasa daerah
    "aneuk": "anak"
}
# Merge, custom_slang override kalau ada konflik
slang_dict.update(custom_slang)

In [7]:
def normalize_slang(text):
    if pd.isna(text) or text == "":
        return ""
    tokens = text.split()
    normalized = [slang_dict.get(word, word) for word in tokens]
    return " ".join(normalized)

In [8]:
def clean_tweet(text):
    if pd.isna(text):
        return ""
    text = str(text)

    hashtags = re.findall(r'#(\w+)', text)
    for ht_word in hashtags:
        expanded = re.sub(r'([a-z])([A-Z])', r'\1 \2', ht_word)
        text = text.replace('#' + ht_word, expanded)

    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+', '', text)

    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text= re.sub(r'(.)\1{2,}', r'\1', text)
    return text

df['clean_text'] = df['full_text'].apply(clean_tweet)
df['clean_text'] = df['clean_text'].apply(normalize_slang)

In [9]:
all_text= " ".join(df['clean_text'].tolist())
all_words=all_text.split()
unique_words=set(all_words)
print(f"Total kata keseluruhan di dataset: {len(all_words)} kata")
print(f"Total kata unik yang perlu di-stem: {len(unique_words)} kata")

Total kata keseluruhan di dataset: 284426 kata
Total kata unik yang perlu di-stem: 22478 kata


In [10]:
stemming_dict={}
for word in tqdm(unique_words, desc="Stemming"):
    if word not in indonesian_stopwords:
        stemmed = stemmer.stem(word)
        if stemmed != "":
            stemming_dict[word]=stemmed
    else:
        stemming_dict[word]= ""

MIN_WORD_LEN = 3

def apply_hash(text):
    tokens = text.split()
    final_tokens = [
        stemming_dict.get(word)
        for word in tokens
        if stemming_dict.get(word) and len(stemming_dict.get(word)) >= MIN_WORD_LEN
    ]
    return " ".join(final_tokens)

df["clean_text"]=df["clean_text"].progress_apply(apply_hash)
display(df[['full_text', 'clean_text']].head())

100%|██████████| 15000/15000 [00:00<00:00, 139582.46it/s]


,full_text,clean_text
0,Surat Edaran (SE) Ditjen Pendis Kemenag Nomor ...,surat edar ditjen pendis kemenag nomor tahun a...
1,PROGRAM MAKAN BERGIZI GRATIS UNTUK ANEUK ACEH\...,anak aceh penuh gizi indonesia indonesiaemas a...
2,@abu_waras @prabowo Sdh pertengahan Januari 20...,tengah januari sekarang list seko masyarakat m...
3,@bahlilngntot @tanyakanrl Agen mbg,agen
4,Program MBG = bukan bisnis. Ini gerakan sosial...,bisnis gera sosial peduli sehat anakanak perca...


In [11]:
def remove_repetition(text):
    tokens = text.split()
    return " ".join(dict.fromkeys(tokens))

df['clean_text'] = df['clean_text'].apply(remove_repetition)

In [12]:
from collections import Counter

MIN_FREQ = 2

all_words_after = " ".join(df['clean_text']).split()
word_freq = Counter(all_words_after)

valid_vocab = {word for word, count in word_freq.items() if count >= MIN_FREQ}

df['clean_text'] = df['clean_text'].apply(
    lambda text: " ".join([w for w in text.split() if w in valid_vocab])
)

print(f"Vocab setelah filter hapax: {len(valid_vocab)} kata unik")

Vocab setelah filter hapax: 6818 kata unik


In [13]:
with open("../Data/Cache/stemming_dict.json", "w") as f:
    json.dump(stemming_dict, f)

# Load kalau udah ada
# with open("stemming_dict.json") as f:
#     stemming_dict = json.load(f)

In [14]:
df.to_csv("../Data/Cleaned/Case 2 Dataset.csv", index=False)